# Saudi Real Estate Market Analysis and Feature Engineering

This notebook loads cleaned real estate, sales, rent, and population datasets, performs data validation and aggregation, and produces city-level market overviews and comparisons for Saudi Arabia.

In [ ]:
# Load standard analytics libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Show all dataframe columns in notebook output
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')


## Load datasets

Load the cleaned master market dataset, sales dataset, rent dataset, and population dataset for analysis and aggregation.

In [ ]:
# Define dataset file paths
master_path = 'cleaned_dataset/master_real_estate_market.csv'
sales_path = 'cleaned_dataset/sales_cleaned.csv'
rent_path = 'cleaned_dataset/rent_cleaned.csv'
population_path = 'cleaned_dataset/population.csv'

# Load cleaned datasets into pandas dataframes
master = pd.read_csv(master_path)
sales = pd.read_csv(sales_path)
rent = pd.read_csv(rent_path)
population = pd.read_csv(population_path)

# Display a quick preview of the master dataset
master.head()


## Initial dataset profile

Inspect the main dataset for shape, missing values, and basic structure.

In [ ]:
# Profile the initial master dataset
print('Master dataset shape:', master.shape)
print('Columns:', master.columns.tolist())
print('Missing values by column:')
print(master.isna().sum())
print('Duplicate rows:', master.duplicated().sum())
print('Unique cities:', master['city'].nunique())
print('Unique regions:', master['region'].nunique())


## Grain and duplicate validation

Verify the expected data grain and inspect duplicate records at the city-quarter-property level.

In [ ]:
# Validate the expected data grain and inspect duplicates
grain_columns = ['city', 'quarter_key', 'property_type']
duplicate_count = master.duplicated(subset=grain_columns).sum()
rows_per_grain = master.groupby(grain_columns).size().max()

print('Duplicate grain rows:', duplicate_count)
print('Maximum rows per grain:', rows_per_grain)

# Calculate record coverage by city
city_coverage = (
    master.groupby('city')
    .agg(first_year=('year', 'min'), last_year=('year', 'max'), quarters_available=('quarter_key', 'nunique'))
    .sort_values('quarters_available')
)
city_coverage.head(15)


## Build the cleaned master dataset from sales and rent data

Aggregate sales and rent datasets using weighted averages and combine them into a final master table.

In [ ]:
# Create weighted aggregates for sales data
sales_work = sales.copy()
sales_work['weighted_price_m2'] = (sales_work['avg_price_m2'] * sales_work['sales_transactions']).where(sales_work['avg_price_m2'].notna(), 0)
sales_work['price_weight'] = sales_work['sales_transactions'].where(sales_work['avg_price_m2'].notna(), 0)

sales_agg = (
    sales_work.groupby(grain_columns, as_index=False, dropna=False)
    .agg(
        sales_transactions=('sales_transactions', 'sum'),
        market_value=('market_value', 'sum'),
        weighted_price_m2=('weighted_price_m2', 'sum'),
        price_weight=('price_weight', 'sum')
    )
)

sales_agg['avg_price_m2'] = np.where(
    sales_agg['price_weight'] > 0,
    sales_agg['weighted_price_m2'] / sales_agg['price_weight'],
    np.nan
)
sales_agg = sales_agg.drop(columns=['weighted_price_m2', 'price_weight'])

# Create weighted aggregates for rent data
rent_work = rent.copy()
rent_work['weighted_rent'] = (rent_work['avg_rent'] * rent_work['rental_contracts']).where(rent_work['avg_rent'].notna(), 0)
rent_work['rent_weight'] = rent_work['rental_contracts'].where(rent_work['avg_rent'].notna(), 0)

rent_agg = (
    rent_work.groupby(grain_columns, as_index=False, dropna=False)
    .agg(
        rental_contracts=('rental_contracts', 'sum'),
        weighted_rent=('weighted_rent', 'sum'),
        rent_weight=('rent_weight', 'sum')
    )
)

rent_agg['avg_rent'] = np.where(
    rent_agg['rent_weight'] > 0,
    rent_agg['weighted_rent'] / rent_agg['rent_weight'],
    np.nan
)
rent_agg = rent_agg.drop(columns=['weighted_rent', 'rent_weight'])

# Merge sales and rent aggregates into the final master dataset
master = sales_agg.merge(rent_agg, on=grain_columns, how='outer', validate='one_to_one')

# Extract year and quarter from the quarter key
master['year'] = master['quarter_key'].str[:4].astype(int)
master['quarter'] = master['quarter_key'].str.extract(r'Q([1-4])', expand=False).astype(int)

city_region_ar_map = {
    "Riyadh": "منطقة الرياض",
    "Jeddah": "منطقة مكة المكرمة",
    "Makkah": "منطقة مكة المكرمة",
    "Madinah": "منطقة المدينة المنورة",
    "Dammam": "المنطقة الشرقية",
    "Khobar": "المنطقة الشرقية",
    "Abha": "منطقة عسير",
    "Jazan": "منطقة جازان",
    "Najran": "منطقة نجران",
    "Tabuk": "منطقة تبوك",
    "Hail": "منطقة حائل",
    "Buraidah": "منطقة القصيم",
    "Sakaka": "منطقة الجوف",
    "Arar": "منطقة الحدود الشمالية",
    "Al Bahah": "منطقة الباحة"
}

city_region_en_map = {
    "Riyadh": "Riyadh Region",
    "Jeddah": "Makkah Region",
    "Makkah": "Makkah Region",
    "Madinah": "Madinah Region",
    "Dammam": "Eastern Province",
    "Khobar": "Eastern Province",
    "Abha": "Aseer Region",
    "Jazan": "Jazan Region",
    "Najran": "Najran Region",
    "Tabuk": "Tabuk Region",
    "Hail": "Hail Region",
    "Buraidah": "Al Qassim Region",
    "Sakaka": "Al Jouf Region",
    "Arar": "Northern Borders Region",
    "Al Bahah": "Al Bahah Region"
}

master["region_ar"] = master["city"].map(city_region_ar_map)
master["region_en"] = master["city"].map(city_region_en_map)

print("Unmapped Arabic regions:", master.loc[master["region_ar"].isna(), "city"].unique())
print("Unmapped English regions:", master.loc[master["region_en"].isna(), "city"].unique())

assert master["region_ar"].notna().all()
assert master["region_en"].notna().all()


# Add population data by city
assert population['city'].is_unique
master = master.merge(population[['city', 'population']], on='city', how='left', validate='many_to_one')

print('Final master shape:', master.shape)
print('Final duplicates by grain:', master.duplicated(subset=grain_columns).sum())


In [ ]:
master

## Export the cleaned master dataset

Save the combined master dataset for downstream analysis and modeling.

In [ ]:
# Export the cleaned master dataset for future analysis
master.to_csv('cleaned_dataset/master_real_estate_final.csv', index=False)
print('Saved cleaned master dataset to cleaned_dataset/master_real_estate_final.csv')


## Missing values in the final master dataset

Review missing values after combining sales, rent, and population data.

In [ ]:
# Calculate missing values after merging all source data
missing = master.isna().sum().to_frame('missing_count')
missing['missing_pct'] = missing['missing_count'] / len(master) * 100
missing.sort_values('missing_pct', ascending=False)


## Sales and rental coverage by year

Inspect time coverage for sales- and rent-related records in the final master table.

In [ ]:
# Analyze coverage for records with sales and rental information
sales_data = master[master['sales_transactions'].notna()].copy()
rent_data = master[master['rental_contracts'].notna()].copy()

print('Sales years:', sales_data['year'].min(), '-', sales_data['year'].max())
print('Rent years:', rent_data['year'].min(), '-', rent_data['year'].max())

display(sales_data.groupby('year')['quarter_key'].nunique().to_frame('sales_quarters'))
display(rent_data.groupby('year')['quarter_key'].nunique().to_frame('rent_quarters'))


## City-level sales overview

Create aggregated city-level sales metrics and population-adjusted indicators.

In [ ]:
# Build a city-level overview with population-adjusted metrics
city_overview = (
    master.groupby('city', as_index=False)
    .agg(
        sales_transactions=('sales_transactions', 'sum'),
        market_value=('market_value', 'sum'),
        rental_contracts=('rental_contracts', 'sum'),
        population=('population', 'max')
    )
)

city_overview['market_value_billion'] = city_overview['market_value'] / 1_000_000_000
city_overview['sales_per_1000_pop'] = city_overview['sales_transactions'] / city_overview['population'] * 1000
city_overview['rentals_per_1000_pop'] = city_overview['rental_contracts'] / city_overview['population'] * 1000
city_overview = city_overview.sort_values('sales_transactions', ascending=False)
display(city_overview.head(20))


In [ ]:
# Plot total sales transactions by city
plot_data = city_overview.sort_values('sales_transactions')
plt.figure(figsize=(10, 7))
plt.barh(plot_data['city'], plot_data['sales_transactions'], color='tab:blue')
plt.title('Total Sales Transactions by City')
plt.xlabel('Sales Transactions')
plt.ylabel('City')
plt.tight_layout()
plt.show()


In [ ]:
# Plot total rental contracts by city
plot_data = city_overview.sort_values('rental_contracts')
plt.figure(figsize=(10, 7))
plt.barh(plot_data['city'], plot_data['rental_contracts'], color='tab:green')
plt.title('Total Rental Contracts by City')
plt.xlabel('Rental Contracts')
plt.ylabel('City')
plt.tight_layout()
plt.show()


## Sales market snapshot for 2018–2022 and 2019–2022

Compare city-level sales activity for the full 2018–2022 period and the 2019–2022 subset.

In [ ]:
# Aggregate sales metrics for the 2018–2022 period
sales_snapshot = master[(master['sales_transactions'].notna()) & (master['year'] <= 2022)].copy()
sales_city = (
    sales_snapshot.groupby('city', as_index=False)
    .agg(
        sales_transactions=('sales_transactions', 'sum'),
        market_value=('market_value', 'sum'),
        population=('population', 'max')
    )
)
sales_city['market_value_billion'] = sales_city['market_value'] / 1_000_000_000
sales_city['avg_transaction_value'] = sales_city['market_value'] / sales_city['sales_transactions']
sales_city['sales_per_1000_pop'] = sales_city['sales_transactions'] / sales_city['population'] * 1000
sales_city = sales_city.sort_values('sales_transactions', ascending=False)
display(sales_city.head(20))


In [ ]:
plot_data = sales_city.sort_values('sales_transactions')
plt.figure(figsize=(10, 7))
plt.barh(plot_data['city'], plot_data['sales_transactions'], color='tab:purple')
plt.title('Sales Transactions by City — 2018–2022')
plt.xlabel('Sales Transactions')
plt.ylabel('City')
plt.tight_layout()
plt.show()


In [ ]:
# Aggregate sales metrics for the 2019–2022 subset
sales_snapshot_2019 = master[(master['sales_transactions'].notna()) & (master['year'].between(2019, 2022))].copy()
sales_city_2019 = (
    sales_snapshot_2019.groupby('city', as_index=False)
    .agg(
        sales_transactions=('sales_transactions', 'sum'),
        market_value=('market_value', 'sum'),
        population=('population', 'max')
    )
)
sales_city_2019['market_value_billion'] = sales_city_2019['market_value'] / 1_000_000_000
sales_city_2019['avg_transaction_value'] = sales_city_2019['market_value'] / sales_city_2019['sales_transactions']
sales_city_2019['sales_per_1000_pop'] = sales_city_2019['sales_transactions'] / sales_city_2019['population'] * 1000
sales_city_2019 = sales_city_2019.sort_values('sales_transactions', ascending=False)
display(sales_city_2019.head(20))


In [ ]:
plot_data = sales_city_2019.sort_values('sales_transactions')
plt.figure(figsize=(10, 7))
plt.barh(plot_data['city'], plot_data['sales_transactions'], color='tab:orange')
plt.title('Sales Transactions by City — 2019–2022')
plt.xlabel('Sales Transactions')
plt.ylabel('City')
plt.tight_layout()
plt.show()


In [ ]:
plot_data = sales_city_2019.sort_values('market_value_billion')
plt.figure(figsize=(10, 7))
plt.barh(plot_data['city'], plot_data['market_value_billion'], color='tab:red')
plt.title('Sales Market Value by City — 2019–2022')
plt.xlabel('Market Value — SAR Billion')
plt.ylabel('City')
plt.tight_layout()
plt.show()


In [ ]:
plot_data = sales_city_2019.sort_values('avg_transaction_value')
plt.figure(figsize=(10, 7))
plt.barh(plot_data['city'], plot_data['avg_transaction_value'], color='tab:brown')
plt.title('Average Transaction Value by City — 2019–2022')
plt.xlabel('Average Transaction Value — SAR')
plt.ylabel('City')
plt.tight_layout()
plt.show()


In [ ]:
plot_data = sales_city_2019.sort_values('sales_per_1000_pop')
plt.figure(figsize=(10, 7))
plt.barh(plot_data['city'], plot_data['sales_per_1000_pop'], color='tab:cyan')
plt.title('Sales Transactions per 1,000 Population — 2019–2022')
plt.xlabel('Sales Transactions per 1,000 Population')
plt.ylabel('City')
plt.tight_layout()
plt.show()


## Rental market overview 2019–2024

Aggregate rental activity and rental unit intensity for cities with rent data.

In [ ]:
# Aggregate rental metrics for the 2019–2024 period
rent_snapshot = master[(master['rental_contracts'].notna()) & (master['year'].between(2019, 2024))].copy()
rent_snapshot['rent_weighted_value'] = rent_snapshot['avg_rent'] * rent_snapshot['rental_contracts']
rent_city = (
    rent_snapshot.groupby('city', as_index=False)
    .agg(
        rental_contracts=('rental_contracts', 'sum'),
        rent_weighted_value=('rent_weighted_value', 'sum'),
        population=('population', 'max')
    )
)
rent_city['avg_rent'] = rent_city['rent_weighted_value'] / rent_city['rental_contracts']
rent_city['rentals_per_1000_pop'] = rent_city['rental_contracts'] / rent_city['population'] * 1000
rent_city = rent_city.drop(columns=['rent_weighted_value']).sort_values('rental_contracts', ascending=False)
display(rent_city.head(20))


In [ ]:
plot_data = rent_city.sort_values('rental_contracts')
plt.figure(figsize=(10, 7))
plt.barh(plot_data['city'], plot_data['rental_contracts'], color='tab:olive')
plt.title('Rental Contracts by City — 2019–2024')
plt.xlabel('Rental Contracts')
plt.ylabel('City')
plt.tight_layout()
plt.show()


In [ ]:
plot_data = rent_city.sort_values('avg_rent')
plt.figure(figsize=(10, 7))
plt.barh(plot_data['city'], plot_data['avg_rent'], color='tab:pink')
plt.title('Average Rent by City — 2019–2024')
plt.xlabel('Average Rent — SAR')
plt.ylabel('City')
plt.tight_layout()
plt.show()


In [ ]:
plot_data = rent_city.sort_values('rentals_per_1000_pop')
plt.figure(figsize=(10, 7))
plt.barh(plot_data['city'], plot_data['rentals_per_1000_pop'], color='tab:gray')
plt.title('Rental Contracts per 1,000 Population — 2019–2024')
plt.xlabel('Rental Contracts per 1,000 Population')
plt.ylabel('City')
plt.tight_layout()
plt.show()


## City comparison for 2019–2022

Compare sales and rental intensity for cities over the 2019–2022 period.

In [ ]:
# Build a city-level comparison dataset for 2019–2022
common_market = master[master['year'].between(2019, 2022)].copy()
common_market['rent_weighted_value'] = common_market['avg_rent'] * common_market['rental_contracts']

city_comparison = (
    common_market.groupby('city', as_index=False)
    .agg(
        sales_transactions=('sales_transactions', 'sum'),
        market_value=('market_value', 'sum'),
        rental_contracts=('rental_contracts', 'sum'),
        rent_weighted_value=('rent_weighted_value', 'sum'),
        population=('population', 'max')
    )
)

city_comparison['avg_rent'] = city_comparison['rent_weighted_value'] / city_comparison['rental_contracts']
city_comparison['market_value_billion'] = city_comparison['market_value'] / 1_000_000_000
city_comparison['sales_per_1000_pop'] = city_comparison['sales_transactions'] / city_comparison['population'] * 1000
city_comparison['rentals_per_1000_pop'] = city_comparison['rental_contracts'] / city_comparison['population'] * 1000
city_comparison = city_comparison.drop(columns=['rent_weighted_value'])
display(city_comparison.sort_values('sales_transactions', ascending=False).head(20))


In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(city_comparison['sales_per_1000_pop'], city_comparison['rentals_per_1000_pop'], alpha=0.8)
for _, row in city_comparison.iterrows():
    plt.text(row['sales_per_1000_pop'], row['rentals_per_1000_pop'], row['city'], fontsize=8)

plt.title('Sales vs Rental Activity per 1,000 Population — 2019–2022')
plt.xlabel('Sales Transactions per 1,000 Population')
plt.ylabel('Rental Contracts per 1,000 Population')
plt.tight_layout()
plt.show()


## Market segmentation by sales and rentals

Classify cities into four segments based on sales and rental intensity relative to median values.

In [ ]:
# Segment cities based on sales and rental activity relative to medians
sales_median = city_comparison['sales_per_1000_pop'].median()
rentals_median = city_comparison['rentals_per_1000_pop'].median()

def classify_market(row):
    if row['sales_per_1000_pop'] >= sales_median and row['rentals_per_1000_pop'] >= rentals_median:
        return 'High Sales / High Rentals'
    if row['sales_per_1000_pop'] >= sales_median and row['rentals_per_1000_pop'] < rentals_median:
        return 'High Sales / Lower Rentals'
    if row['sales_per_1000_pop'] < sales_median and row['rentals_per_1000_pop'] >= rentals_median:
        return 'Lower Sales / High Rentals'
    return 'Lower Sales / Lower Rentals'

city_comparison['market_segment'] = city_comparison.apply(classify_market, axis=1)

display(
    city_comparison[['city', 'sales_per_1000_pop', 'rentals_per_1000_pop', 'market_segment']]
    .sort_values(['market_segment', 'sales_per_1000_pop'], ascending=[True, False])
)


In [ ]:
# Export the city-level comparison dataset to CSV
city_comparison.to_csv('cleaned_dataset/city_market_comparison_2019_2022.csv', index=False, encoding='utf-8-sig')
print('Saved city comparison to cleaned_dataset/city_market_comparison_2019_2022.csv')


## Derived metric definitions

- **market_value_billion**: total market value converted to SAR billions for easier comparison.
- **avg_transaction_value**: average sales value per transaction in each city.
- **sales_per_1000_pop**: sales transactions normalized to a per-1,000-population basis.
- **rentals_per_1000_pop**: rental contracts normalized to a per-1,000-population basis.
- **avg_rent**: weighted average rent calculated using rental contract volume.
- **market_segment**: categorical grouping of cities based on sales and rental intensity relative to medians.


## Derived metric definitions

- **market_value_billion**: total market value converted to SAR billions for easier comparison.
- **avg_transaction_value**: average sales value per transaction in each city.
- **sales_per_1000_pop**: sales transactions normalized to a per-1,000-population basis.
- **rentals_per_1000_pop**: rental contracts normalized to a per-1,000-population basis.
- **avg_rent**: weighted average rent calculated using rental contract volume.
- **market_segment**: categorical grouping of cities based on sales and rental intensity relative to medians.



## Summary and next steps

- Combined sales, rent, and population data into a single cleaned master dataset.
- Validated grain consistency and checked for duplicate city-quarter-property rows.
- Created city-level performance metrics for sales, rental activity, and population-adjusted intensity.
- Compared city market activity over key time windows and classified cities into four market segments.

### Recommended next steps

1. Review missing value counts and investigate any cities with incomplete population or pricing data.
2. Add time-series trend charts for top cities to highlight market momentum.
3. Use the cleaned dataset for modeling, forecasting, or Power BI dashboard creation.
